In [7]:
# this script merges the raw data until the previous quarter with the file for the actual quarter


import pandas as pd
import os

input1 = "../../50 KM Group/Royalties/Statements/Karen/Rock Music/Consolidated statements/old/Rock_royalties_2021Q1_2025Q4_1_raw_combined.csv"
input2 = "../../50 KM Group/Royalties/Statements/Karen/Rock Music/Quarterly statements/2026 Q1/MOK 2026Q1_JN.xlsx"
outputfilename = "../../50 KM Group/Royalties/Statements/Karen/_output/Rock_royalties_2021Q1_2026Q1_1_raw_combined.csv"

df_1 = pd.read_csv(input1,low_memory=False)

print(f"DataFrame: Rows: {df_1.shape[0]}, Columns: {df_1.shape[1]}")
print(f"Total fee: {df_1['SHARE AMOUNT (local FX)'].sum()}. Total units: {df_1['UNIT'].sum()}")

df_2 = pd.read_excel(input2,sheet_name = "data")
print(f"DataFrame: Rows: {df_2.shape[0]}, Columns: {df_2.shape[1]}")
df_2["CATALOG NO._MOD"] = "mod."+ df_2["CATALOG NO."]
print(f"Total fee: {df_2['SHARE AMOUNT (local FX)'].sum()}. Total units: {df_2['UNIT'].sum()}")

def align_columns(df1, df2):
    missing_in_df1 = df2.columns.difference(df1.columns)
    missing_in_df2 = df1.columns.difference(df2.columns)    
    
    for col in missing_in_df1:
        df1[col] = pd.NA
    for col in missing_in_df2:
        df2[col] = pd.NA
        
    return df1, df2

def convert_to_majority_type(df):
    df_copy = df.copy()
    
    for col in df_copy.columns:
        # Get the majority type
        type_counts = df_copy[col].map(type).value_counts()
        majority_type = type_counts.idxmax()
        
        # Define the conversion function
        if majority_type == str:
            df_copy[col] = df_copy[col].astype(str)
        elif majority_type == float:
            df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')
        elif majority_type == int:
            df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce').astype('Int64')
        elif majority_type == bool:
            df_copy[col] = df_copy[col].astype(bool)
        else:
            # Fallback: use string conversion
            df_copy[col] = df_copy[col].astype(str)
    
    return df_copy


print('columns in df1:')
for column in df_1.columns:        
        print(column)

print('columns in df2:')
for column in df_2.columns:        
        print(column)


DataFrame: Rows: 1214707, Columns: 29
Total fee: 6444701.433998934. Total units: 6966629757
DataFrame: Rows: 184198, Columns: 28
Total fee: 432486.8694841999. Total units: 386869455
columns in df1:
AMOUNT
ARTIST
Base Price
CATALOG NO.
CATALOG NO._MOD
CATALOG TITLE
Ctrl.%
Currency
Entry No.
PayType
Payee/Licensor
Payer/Licensee
REVENUE PERIOD
ROYALTY
Release Date
Report Quarter
Report year
Rev Quarter
Rev Year
Royalty Rate%
SHARE AMOUNT (local FX)
SONG TITLE
Share%
SongProRata
Territory
Type
UNIT
USER
WS Price
columns in df2:
Entry No.
Payer/Licensee
Payee/Licensor
Territory
Report year
Report Quarter
Rev Year
Rev Quarter
USER
CATALOG NO.
PayType
CATALOG TITLE
SONG TITLE
ARTIST
REVENUE PERIOD
UNIT
AMOUNT
SongProRata
Ctrl.%
Royalty Rate%
ROYALTY
Share%
SHARE AMOUNT (local FX)
Currency
WS Price
Base Price
Release Date
Type
CATALOG NO._MOD


In [8]:
# print(df_1["Statement Quarter"].map(type).value_counts())

def clean_up_dtype (df,df_name):
    print(f"\nChecking what data types there are in {df_name}")
    for col in df.columns:
        print(f"Column: {col}")
        print(df[col].map(type).value_counts())
        print()
    dfc = convert_to_majority_type(df)
    print(f"\nConverting data types in {df_name}")
    for col in dfc.columns:
        print(f"Column: {col}")
        print(dfc[col].map(type).value_counts())
        print()
    return dfc

df_1c = clean_up_dtype(df_1,input1)
df_2c = clean_up_dtype(df_2,input2)

#df_2c["Sales Month"] = df_2c["Sales Month"].astype("string")



Checking what data types there are in ../../50 KM Group/Royalties/Statements/Karen/Rock Music/Consolidated statements/old/Rock_royalties_2021Q1_2025Q4_1_raw_combined.csv
Column: AMOUNT
AMOUNT
<class 'float'>    1214707
Name: count, dtype: int64

Column: ARTIST
ARTIST
<class 'str'>      1214306
<class 'float'>        401
Name: count, dtype: int64

Column: Base Price
Base Price
<class 'float'>    1214707
Name: count, dtype: int64

Column: CATALOG NO.
CATALOG NO.
<class 'str'>    1214707
Name: count, dtype: int64

Column: CATALOG NO._MOD
CATALOG NO._MOD
<class 'str'>    1214707
Name: count, dtype: int64

Column: CATALOG TITLE
CATALOG TITLE
<class 'str'>    1214707
Name: count, dtype: int64

Column: Ctrl.%
Ctrl.%
<class 'int'>    1214707
Name: count, dtype: int64

Column: Currency
Currency
<class 'str'>    1214707
Name: count, dtype: int64

Column: Entry No.
Entry No.
<class 'float'>    1214707
Name: count, dtype: int64

Column: PayType
PayType
<class 'str'>    1214707
Name: count, dtype:

In [9]:
# Find common columns
common_cols = df_1c.columns.intersection(df_2c.columns)

# Dictionary to hold comparison results
type_comparison = {}

# Loop through each common column
for col in common_cols:
    # Get the set of types present in each column (ignoring NaN)
    types_df1 = set(df_1c[col].dropna().map(type))
    types_df2 = set(df_2c[col].dropna().map(type))
    
    # Store in the results dictionary
    type_comparison[col] = {
        "df1_types": types_df1,
        "df2_types": types_df2,
        "types_match": types_df1 == types_df2
    }

# Convert to a DataFrame for nicer display
type_comparison_df = pd.DataFrame(type_comparison).T

print(type_comparison_df)


                                 df1_types          df2_types types_match
AMOUNT                   {<class 'float'>}  {<class 'float'>}        True
ARTIST                     {<class 'str'>}    {<class 'str'>}        True
Base Price               {<class 'float'>}  {<class 'float'>}        True
CATALOG NO.                {<class 'str'>}    {<class 'str'>}        True
CATALOG NO._MOD            {<class 'str'>}    {<class 'str'>}        True
CATALOG TITLE              {<class 'str'>}    {<class 'str'>}        True
Ctrl.%                     {<class 'int'>}    {<class 'int'>}        True
Currency                   {<class 'str'>}    {<class 'str'>}        True
Entry No.                {<class 'float'>}  {<class 'float'>}        True
PayType                    {<class 'str'>}    {<class 'str'>}        True
Payee/Licensor             {<class 'str'>}    {<class 'str'>}        True
Payer/Licensee             {<class 'str'>}    {<class 'str'>}        True
REVENUE PERIOD             {<class 'st

In [10]:
df_1c, df_2c = align_columns(df_1c, df_2c)
df_final = pd.concat([df_1c, df_2c], ignore_index=True)
print(f"Total fee: {df_final['SHARE AMOUNT (local FX)'].sum()}. Total units: {df_final['UNIT'].sum()}")

df_final = df_final.sort_index(axis=1)

print('columns in df_final:')
for column in df_final.columns:        
        print(column)

print(f"Merged DataFrame: Rows: {df_final.shape[0]}, Columns: {df_final.shape[1]}")
df_final = df_final.loc[:, ~df_final.columns.str.contains("^Unnamed")]
print(f"Merged DataFrame: Rows: {df_final.shape[0]}, Columns: {df_final.shape[1]}")



Total fee: 6877188.30348313. Total units: 7353499212
columns in df_final:
AMOUNT
ARTIST
Base Price
CATALOG NO.
CATALOG NO._MOD
CATALOG TITLE
Ctrl.%
Currency
Entry No.
PayType
Payee/Licensor
Payer/Licensee
REVENUE PERIOD
ROYALTY
Release Date
Report Quarter
Report year
Rev Quarter
Rev Year
Royalty Rate%
SHARE AMOUNT (local FX)
SONG TITLE
Share%
SongProRata
Territory
Type
UNIT
USER
WS Price
Merged DataFrame: Rows: 1398905, Columns: 29
Merged DataFrame: Rows: 1398905, Columns: 29


In [11]:
for col in df_final.columns:
    print(f"Column: {col}")
    print(df_final[col].map(type).value_counts())
    print()

Column: AMOUNT
AMOUNT
<class 'float'>    1398905
Name: count, dtype: int64

Column: ARTIST
ARTIST
<class 'str'>    1398905
Name: count, dtype: int64

Column: Base Price
Base Price
<class 'float'>    1398905
Name: count, dtype: int64

Column: CATALOG NO.
CATALOG NO.
<class 'str'>    1398905
Name: count, dtype: int64

Column: CATALOG NO._MOD
CATALOG NO._MOD
<class 'str'>    1398905
Name: count, dtype: int64

Column: CATALOG TITLE
CATALOG TITLE
<class 'str'>    1398905
Name: count, dtype: int64

Column: Ctrl.%
Ctrl.%
<class 'int'>    1398905
Name: count, dtype: int64

Column: Currency
Currency
<class 'str'>    1398905
Name: count, dtype: int64

Column: Entry No.
Entry No.
<class 'float'>    1398905
Name: count, dtype: int64

Column: PayType
PayType
<class 'str'>    1398905
Name: count, dtype: int64

Column: Payee/Licensor
Payee/Licensor
<class 'str'>    1398905
Name: count, dtype: int64

Column: Payer/Licensee
Payer/Licensee
<class 'str'>    1398905
Name: count, dtype: int64

Column: REVE

In [12]:
df_final.to_csv(outputfilename, index=False)